In [1]:
import numpy as np

In [19]:
neighbors = {
    "Brazil": ["Argentina", "Bolivia", "Paraguay", "Uruguay", "Colombia",
               "Peru", "Venezuela", "Guyana", "Suriname", "French Guiana"],
    "Argentina": ["Chile","Bolivia","Paraguay","Uruguay","Brazil"],
    "Bolivia": ["Chile","Argentina","Paraguay","Brazil","Peru"],
    "Peru": ["Chile","Bolivia","Brazil","Colombia","Ecuador"],
    "Colombia": ["Venezuela","Brazil","Peru","Ecuador"],
    "Chile": ["Argentina","Bolivia","Peru"],
    "Paraguay": ["Argentina","Bolivia","Brazil"],
    "Venezuela": ["Colombia","Brazil","Guyana"],
    "Uruguay": ["Argentina","Brazil"],
    "Ecuador": ["Colombia","Peru"],
    "Guyana": ["Venezuela","Brazil","Suriname"],
    "Suriname": ["Guyana","Brazil","French Guiana"],
    "French Guiana": ["Suriname","Brazil"],
}

countries = list(neighbors.keys())
n   = len(countries)
idx = {c: i for i, c in enumerate(countries)}

print(countries)

['Brazil', 'Argentina', 'Bolivia', 'Peru', 'Colombia', 'Chile', 'Paraguay', 'Venezuela', 'Uruguay', 'Ecuador', 'Guyana', 'Suriname', 'French Guiana']


In [21]:
# === Approach 1 ===
def stationary_analytical(neighbors):
    degrees = {c: len(nb) for c, nb in neighbors.items()}
    total = sum(degrees.values())
    return {c: d / total for c, d in degrees.items()}

In [23]:
# === Approach 2 ===
def build_transition_matrix(neighbors, countries, idx):
    n = len(countries)
    T = np.zeros((n, n))
    for c, nbrs in neighbors.items():
        for nb in nbrs:
            T[idx[c]][idx[nb]] = 1.0 / len(nbrs)
    return T

def stationary_eigenvector(T, countries):
    eigenvalues, eigenvectors = np.linalg.eig(T.T)
    ev_idx = np.argmin(np.abs(eigenvalues - 1.0))
    pi = np.real(eigenvectors[:, ev_idx])
    pi = pi / pi.sum()
    return {countries[i]: pi[i] for i in range(len(countries))}

In [25]:
# === Approach 3 ===
def stationary_power(T, start_country, countries, idx, steps=1000):
    pi = np.zeros(len(countries))
    pi[idx[start_country]] = 1.0
    pi = pi @ np.linalg.matrix_power(T, steps)
    return {countries[i]: pi[i] for i in range(len(countries))}

In [27]:
T = build_transition_matrix(neighbors, countries, idx)

results = {
    "Analytical":         stationary_analytical(neighbors),
    "Eigenvector":        stationary_eigenvector(T, countries),
    "Power iteration":    stationary_power(T, "Chile", countries, idx),
}

for method, dist in results.items():
    ranked = sorted(dist.items(), key=lambda x: -x[1])
    print(f"\n{'─'*50}")
    print(f"  {method}")
    print(f"{'─'*50}")
    for country, prob in ranked:
        bar = "█" * int(prob * 200)
        print(f"  {country:<16} {prob*100:5.2f}%  {bar}")

# Verify all methods agree
pi_a = np.array([results["Analytical"][c]      for c in countries])
pi_e = np.array([results["Eigenvector"][c]     for c in countries])
pi_p = np.array([results["Power iteration"][c] for c in countries])
print(f"\nAll methods agree: {np.allclose(pi_a, pi_e, atol=1e-6) and np.allclose(pi_a, pi_p, atol=1e-6)}")

best = max(results["Analytical"], key=results["Analytical"].get)
prob = results["Analytical"][best]
print(f"\nAnswer: The spy is most likely in {best} ({prob*100:.1f}% probability)")


──────────────────────────────────────────────────
  Analytical
──────────────────────────────────────────────────
  Brazil           20.00%  ████████████████████████████████████████
  Argentina        10.00%  ████████████████████
  Bolivia          10.00%  ████████████████████
  Peru             10.00%  ████████████████████
  Colombia          8.00%  ████████████████
  Chile             6.00%  ████████████
  Paraguay          6.00%  ████████████
  Venezuela         6.00%  ████████████
  Guyana            6.00%  ████████████
  Suriname          6.00%  ████████████
  Uruguay           4.00%  ████████
  Ecuador           4.00%  ████████
  French Guiana     4.00%  ████████

──────────────────────────────────────────────────
  Eigenvector
──────────────────────────────────────────────────
  Brazil           20.00%  ███████████████████████████████████████
  Peru             10.00%  ████████████████████
  Bolivia          10.00%  ████████████████████
  Argentina        10.00%  █████████████